# ANSI SQL Using MySQL — Module 1
## 25 Exercises with Query Output

**Database:** `event_management`  
**Tables:** `Users` · `Events` · `Sessions` · `Registrations` · `Feedback` · `Resources`

> Run cells top-to-bottom. Update **`DB_USER`** and **`DB_PASSWORD`** in the Config cell.

In [ ]:
import sys
!{sys.executable} -m pip install mysql-connector-python pandas --quiet

In [ ]:
import mysql.connector
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
# ── Configure your MySQL connection ──────────────────────────
DB_HOST     = 'localhost'
DB_USER     = 'root'
DB_PASSWORD = ''          # ← change if you have a password
# ─────────────────────────────────────────────────────────────

conn = mysql.connector.connect(
    host=DB_HOST, user=DB_USER, password=DB_PASSWORD, autocommit=True
)
print('Connected to MySQL ✓')

## Database Setup
Creates the `event_management` database, all 6 tables, and inserts sample data.  
Safe to re-run — uses `IF NOT EXISTS` / `INSERT IGNORE`.

In [ ]:
cursor = conn.cursor()
statements = [
    """CREATE DATABASE IF NOT EXISTS event_management;""",
    """USE event_management;""",
    """CREATE TABLE IF NOT EXISTS Users (
    user_id           INT          PRIMARY KEY AUTO_INCREMENT,
    full_name         VARCHAR(100) NOT NULL,
    email             VARCHAR(100) NOT NULL UNIQUE,
    city              VARCHAR(100) NOT NULL,
    registration_date DATE         NOT NULL
);""",
    """CREATE TABLE IF NOT EXISTS Events (
    event_id     INT          PRIMARY KEY AUTO_INCREMENT,
    title        VARCHAR(200) NOT NULL,
    description  TEXT,
    city         VARCHAR(100) NOT NULL,
    start_date   DATETIME     NOT NULL,
    end_date     DATETIME     NOT NULL,
    status       ENUM('upcoming','completed','cancelled'),
    organizer_id INT,
    CONSTRAINT fk_events_organizer FOREIGN KEY (organizer_id) REFERENCES Users(user_id)
);""",
    """CREATE TABLE IF NOT EXISTS Sessions (
    session_id   INT          PRIMARY KEY AUTO_INCREMENT,
    event_id     INT,
    title        VARCHAR(200) NOT NULL,
    speaker_name VARCHAR(100) NOT NULL,
    start_time   DATETIME     NOT NULL,
    end_time     DATETIME     NOT NULL,
    CONSTRAINT fk_sessions_event FOREIGN KEY (event_id) REFERENCES Events(event_id)
);""",
    """CREATE TABLE IF NOT EXISTS Registrations (
    registration_id   INT  PRIMARY KEY AUTO_INCREMENT,
    user_id           INT,
    event_id          INT,
    registration_date DATE NOT NULL,
    CONSTRAINT fk_reg_user  FOREIGN KEY (user_id)  REFERENCES Users(user_id),
    CONSTRAINT fk_reg_event FOREIGN KEY (event_id) REFERENCES Events(event_id)
);""",
    """CREATE TABLE IF NOT EXISTS Feedback (
    feedback_id   INT  PRIMARY KEY AUTO_INCREMENT,
    user_id       INT,
    event_id      INT,
    rating        INT  CHECK (rating BETWEEN 1 AND 5),
    comments      TEXT,
    feedback_date DATE NOT NULL,
    CONSTRAINT fk_feedback_user  FOREIGN KEY (user_id)  REFERENCES Users(user_id),
    CONSTRAINT fk_feedback_event FOREIGN KEY (event_id) REFERENCES Events(event_id)
);""",
    """CREATE TABLE IF NOT EXISTS Resources (
    resource_id   INT          PRIMARY KEY AUTO_INCREMENT,
    event_id      INT,
    resource_type ENUM('pdf','image','link'),
    resource_url  VARCHAR(255) NOT NULL,
    uploaded_at   DATETIME     NOT NULL,
    CONSTRAINT fk_resources_event FOREIGN KEY (event_id) REFERENCES Events(event_id)
);""",
    """INSERT IGNORE INTO Users VALUES
(1,'Alice Johnson','alice@example.com','New York','2024-12-01'),
(2,'Bob Smith','bob@example.com','Los Angeles','2024-12-05'),
(3,'Charlie Lee','charlie@example.com','Chicago','2024-12-10'),
(4,'Diana King','diana@example.com','New York','2025-01-15'),
(5,'Ethan Hunt','ethan@example.com','Los Angeles','2025-02-01');""",
    """INSERT IGNORE INTO Events VALUES
(1,'Tech Innovators Meetup','A meetup for tech enthusiasts.','New York','2025-06-10 10:00:00','2025-06-10 16:00:00','upcoming',1),
(2,'AI & ML Conference','Conference on AI and ML advancements.','Chicago','2025-05-15 09:00:00','2025-05-15 17:00:00','completed',3),
(3,'Frontend Development Bootcamp','Hands-on training on frontend tech.','Los Angeles','2025-07-01 10:00:00','2025-07-03 16:00:00','upcoming',2);""",
    """INSERT IGNORE INTO Sessions VALUES
(1,1,'Opening Keynote','Dr. Tech','2025-06-10 10:00:00','2025-06-10 11:00:00'),
(2,1,'Future of Web Dev','Alice Johnson','2025-06-10 11:15:00','2025-06-10 12:30:00'),
(3,2,'AI in Healthcare','Charlie Lee','2025-05-15 09:30:00','2025-05-15 11:00:00'),
(4,3,'Intro to HTML5','Bob Smith','2025-07-01 10:00:00','2025-07-01 12:00:00');""",
    """INSERT IGNORE INTO Registrations VALUES
(1,1,1,'2025-05-01'),(2,2,1,'2025-05-02'),
(3,3,2,'2025-04-30'),(4,4,2,'2025-04-28'),
(5,5,3,'2025-06-15');""",
    """INSERT IGNORE INTO Feedback VALUES
(1,3,2,4,'Great insights!','2025-05-16'),
(2,4,2,5,'Very informative.','2025-05-16'),
(3,2,1,3,'Could be better.','2025-06-11');""",
    """INSERT IGNORE INTO Resources VALUES
(1,1,'pdf','https://portal.com/resources/tech_meetup_agenda.pdf','2025-05-01 10:00:00'),
(2,2,'image','https://portal.com/resources/ai_poster.jpg','2025-04-20 09:00:00'),
(3,3,'link','https://portal.com/resources/html5_docs','2025-06-25 15:00:00');""",
]
for stmt in statements:
    try:
        cursor.execute(stmt)
    except mysql.connector.Error as e:
        print(f'  [warn] {e}')
print('Schema ready ✓')

## Helper Function

In [ ]:
def run_query(sql, title=''):
    """Execute sql against event_management and display as DataFrame."""
    conn.database = 'event_management'
    cursor = conn.cursor()
    cursor.execute(sql)
    cols = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    df   = pd.DataFrame(rows, columns=cols)
    if df.empty:
        print('  (no rows returned)')
    else:
        display(df)
    return df

---
## Exercises

### Exercise 1: User Upcoming Events
Show all upcoming events a user is registered for **in their city**, sorted by start date.

In [ ]:
sql = """
SELECT
    u.user_id, u.full_name,
    e.title AS event_title, e.city, e.start_date
FROM Users u
JOIN Registrations r ON r.user_id  = u.user_id
JOIN Events        e ON e.event_id = r.event_id
WHERE e.status = 'upcoming'
  AND e.city   = u.city
ORDER BY e.start_date
"""
print(sql)
run_query(sql)

### Exercise 2: Top Rated Events
Events with the highest average rating, having **at least 10** feedback submissions.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    ROUND(AVG(f.rating), 2)    AS avg_rating,
    COUNT(f.feedback_id)       AS feedback_count
FROM Events   e
JOIN Feedback f ON f.event_id = e.event_id
GROUP BY e.event_id, e.title
HAVING COUNT(f.feedback_id) >= 10
ORDER BY avg_rating DESC
"""
print(sql)
run_query(sql)

### Exercise 3: Inactive Users
Users who have **not registered** for any event in the last 90 days.

In [ ]:
sql = """
SELECT u.user_id, u.full_name, u.email
FROM Users u
WHERE u.user_id NOT IN (
    SELECT r.user_id
    FROM   Registrations r
    WHERE  r.registration_date >= CURDATE() - INTERVAL 90 DAY
)
"""
print(sql)
run_query(sql)

### Exercise 4: Peak Session Hours
Count sessions scheduled between **10 AM and 12 PM** for each event.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    COUNT(s.session_id) AS peak_hour_sessions
FROM Events   e
JOIN Sessions s ON s.event_id = e.event_id
WHERE HOUR(s.start_time) >= 10
  AND HOUR(s.start_time) <  12
GROUP BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 5: Most Active Cities
Top 5 cities with the highest number of **distinct user registrations**.

In [ ]:
sql = """
SELECT
    u.city,
    COUNT(DISTINCT r.user_id) AS distinct_registrations
FROM Users         u
JOIN Registrations r ON r.user_id = u.user_id
GROUP BY u.city
ORDER BY distinct_registrations DESC
LIMIT 5
"""
print(sql)
run_query(sql)

### Exercise 6: Event Resource Summary
Number of PDFs, images, and links uploaded per event.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    SUM(CASE WHEN r.resource_type = 'pdf'   THEN 1 ELSE 0 END) AS pdf_count,
    SUM(CASE WHEN r.resource_type = 'image' THEN 1 ELSE 0 END) AS image_count,
    SUM(CASE WHEN r.resource_type = 'link'  THEN 1 ELSE 0 END) AS link_count,
    COUNT(r.resource_id)                                        AS total_resources
FROM Events    e
LEFT JOIN Resources r ON r.event_id = e.event_id
GROUP BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 7: Low Feedback Alerts
Users who gave a rating **< 3**, with their comments and event name.

In [ ]:
sql = """
SELECT
    u.user_id, u.full_name,
    e.title AS event_title,
    f.rating, f.comments, f.feedback_date
FROM Feedback f
JOIN Users  u ON u.user_id  = f.user_id
JOIN Events e ON e.event_id = f.event_id
WHERE f.rating < 3
"""
print(sql)
run_query(sql)

### Exercise 8: Sessions per Upcoming Event
All upcoming events with their session count.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    COUNT(s.session_id) AS session_count
FROM Events e
LEFT JOIN Sessions s ON s.event_id = e.event_id
WHERE e.status = 'upcoming'
GROUP BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 9: Organizer Event Summary
For each organizer, number of events per status.

In [ ]:
sql = """
SELECT
    u.user_id   AS organizer_id,
    u.full_name AS organizer_name,
    e.status,
    COUNT(e.event_id) AS event_count
FROM Users  u
JOIN Events e ON e.organizer_id = u.user_id
GROUP BY u.user_id, u.full_name, e.status
ORDER BY u.user_id, e.status
"""
print(sql)
run_query(sql)

### Exercise 10: Feedback Gap
Events that had registrations but received **no feedback** at all.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    COUNT(DISTINCT r.registration_id) AS total_registrations
FROM Events        e
JOIN Registrations r  ON r.event_id = e.event_id
LEFT JOIN Feedback f  ON f.event_id = e.event_id
WHERE f.feedback_id IS NULL
GROUP BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 11: Daily New User Count
Number of new users registered **each day in the last 7 days**.

In [ ]:
sql = """
SELECT
    registration_date,
    COUNT(user_id) AS new_users
FROM Users
WHERE registration_date >= CURDATE() - INTERVAL 7 DAY
GROUP BY registration_date
ORDER BY registration_date
"""
print(sql)
run_query(sql)

### Exercise 12: Event with Maximum Sessions
Event(s) with the **highest** number of sessions.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    COUNT(s.session_id) AS session_count
FROM Events   e
JOIN Sessions s ON s.event_id = e.event_id
GROUP BY e.event_id, e.title
HAVING COUNT(s.session_id) = (
    SELECT MAX(cnt)
    FROM (SELECT COUNT(session_id) AS cnt
          FROM Sessions GROUP BY event_id) AS sub
)
"""
print(sql)
run_query(sql)

### Exercise 13: Average Rating per City
Average feedback rating of events conducted in each city.

In [ ]:
sql = """
SELECT
    e.city,
    ROUND(AVG(f.rating), 2) AS avg_rating
FROM Events   e
JOIN Feedback f ON f.event_id = e.event_id
GROUP BY e.city
ORDER BY avg_rating DESC
"""
print(sql)
run_query(sql)

### Exercise 14: Most Registered Events
Top 3 events by total registration count.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    COUNT(r.registration_id) AS total_registrations
FROM Events        e
JOIN Registrations r ON r.event_id = e.event_id
GROUP BY e.event_id, e.title
ORDER BY total_registrations DESC
LIMIT 3
"""
print(sql)
run_query(sql)

### Exercise 15: Event Session Time Conflict
Pairs of sessions within the **same event** whose time ranges overlap.

In [ ]:
sql = """
SELECT
    s1.event_id,
    s1.session_id  AS session_a, s1.title AS title_a,
    s1.start_time  AS start_a,  s1.end_time AS end_a,
    s2.session_id  AS session_b, s2.title AS title_b,
    s2.start_time  AS start_b,  s2.end_time AS end_b
FROM Sessions s1
JOIN Sessions s2
  ON  s1.event_id   = s2.event_id
  AND s1.session_id < s2.session_id
  AND s1.start_time < s2.end_time
  AND s1.end_time   > s2.start_time
"""
print(sql)
run_query(sql)

### Exercise 16: Unregistered Active Users
Users who signed up in the **last 30 days** but haven't registered for any event.

In [ ]:
sql = """
SELECT
    u.user_id, u.full_name, u.registration_date
FROM Users u
WHERE u.registration_date >= CURDATE() - INTERVAL 30 DAY
  AND u.user_id NOT IN (
      SELECT DISTINCT user_id FROM Registrations
  )
"""
print(sql)
run_query(sql)

### Exercise 17: Multi-Session Speakers
Speakers handling **more than one session** across all events.

In [ ]:
sql = """
SELECT
    speaker_name,
    COUNT(session_id) AS session_count
FROM Sessions
GROUP BY speaker_name
HAVING COUNT(session_id) > 1
ORDER BY session_count DESC
"""
print(sql)
run_query(sql)

### Exercise 18: Resource Availability Check
Events that have **no resources** uploaded.

In [ ]:
sql = """
SELECT
    e.event_id, e.title
FROM Events e
LEFT JOIN Resources r ON r.event_id = e.event_id
WHERE r.resource_id IS NULL
"""
print(sql)
run_query(sql)

### Exercise 19: Completed Events with Feedback Summary
For completed events: total registrations and average feedback rating.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    COUNT(DISTINCT r.registration_id) AS total_registrations,
    ROUND(AVG(f.rating), 2)           AS avg_rating
FROM Events e
LEFT JOIN Registrations r ON r.event_id = e.event_id
LEFT JOIN Feedback      f ON f.event_id = e.event_id
WHERE e.status = 'completed'
GROUP BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 20: User Engagement Index
For each user: events registered and feedbacks submitted.

In [ ]:
sql = """
SELECT
    u.user_id, u.full_name,
    COUNT(DISTINCT r.event_id)    AS events_registered,
    COUNT(DISTINCT f.feedback_id) AS feedbacks_submitted
FROM Users u
LEFT JOIN Registrations r ON r.user_id = u.user_id
LEFT JOIN Feedback      f ON f.user_id = u.user_id
GROUP BY u.user_id, u.full_name
ORDER BY events_registered DESC, feedbacks_submitted DESC
"""
print(sql)
run_query(sql)

### Exercise 21: Top Feedback Providers
Top 5 users by number of feedback entries submitted.

In [ ]:
sql = """
SELECT
    u.user_id, u.full_name,
    COUNT(f.feedback_id) AS feedback_count
FROM Users    u
JOIN Feedback f ON f.user_id = u.user_id
GROUP BY u.user_id, u.full_name
ORDER BY feedback_count DESC
LIMIT 5
"""
print(sql)
run_query(sql)

### Exercise 22: Duplicate Registrations Check
Detect users registered **more than once** for the same event.

In [ ]:
sql = """
SELECT
    user_id, event_id,
    COUNT(*) AS registration_count
FROM Registrations
GROUP BY user_id, event_id
HAVING COUNT(*) > 1
"""
print(sql)
run_query(sql)

### Exercise 23: Registration Trends
Month-wise registration count over the **past 12 months**.

In [ ]:
sql = """
SELECT
    DATE_FORMAT(registration_date, '%Y-%m') AS month,
    COUNT(registration_id)                  AS registrations
FROM Registrations
WHERE registration_date >= CURDATE() - INTERVAL 12 MONTH
GROUP BY DATE_FORMAT(registration_date, '%Y-%m')
ORDER BY month
"""
print(sql)
run_query(sql)

### Exercise 24: Average Session Duration per Event
Average duration **(in minutes)** of sessions in each event.

In [ ]:
sql = """
SELECT
    e.event_id, e.title,
    ROUND(AVG(TIMESTAMPDIFF(MINUTE, s.start_time, s.end_time)), 2) AS avg_duration_minutes
FROM Events   e
JOIN Sessions s ON s.event_id = e.event_id
GROUP BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 25: Events Without Sessions
All events that currently have **no sessions** scheduled.

In [ ]:
sql = """
SELECT
    e.event_id, e.title, e.status
FROM Events e
LEFT JOIN Sessions s ON s.event_id = e.event_id
WHERE s.session_id IS NULL
"""
print(sql)
run_query(sql)